# External radiomics feature-handling controls

**Scientific purpose.** Inspect five verified controls separating observed-feature and
training-fold-median contributions.

**Runnability classification:** aggregate reanalysis is runnable from a clean clone. Patient-level
recreation requires private external features, internal development features, splits, and fitted
components.

Conditions A, B, D1, and D2 modify inputs to retained pathways. Condition C is different: it
identifies an externally defined availability subset without labels and refits radiomics classifiers
on internal development data. It is an internal diagnostic refit, not frozen external validation.


In [ ]:
from pathlib import Path
import sys


def locate_repository(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


REPO_ROOT = locate_repository()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import numpy as np
import pandas as pd

CONTROL_DEFINITIONS = {
    "A": "Observed external cells; unavailable cells use internal fold medians.",
    "B": "All selected cells use internal fold medians.",
    "C": "Internal diagnostic refit using features available for every external patient.",
    "D1": "Observed cells retained; unavailable cells set to zero.",
    "D2": "Observed cells zeroed; unavailable cells use internal fold medians.",
}
pd.DataFrame(
    [{"condition": name, "definition": definition} for name, definition in CONTROL_DEFINITIONS.items()]
)


## Deterministic matrix construction

The four direct substitution conditions use fold-training medians only. Strict-overlap Condition C
uses its separately verified internal fitting path and must not be sent to the substitution helper.


In [ ]:
def build_control_matrix(external_values, fold_medians, condition):
    values = np.asarray(external_values, dtype=np.float32)
    medians = np.asarray(fold_medians, dtype=np.float32).reshape(1, -1)
    if values.ndim != 2 or values.shape[1] != medians.shape[1]:
        raise ValueError("External values and fold medians are not feature-aligned.")
    observed = np.isfinite(values)
    if condition == "real_plus_imputed":
        return np.where(observed, values, medians)
    if condition == "median_only":
        return np.broadcast_to(medians, values.shape).copy()
    if condition == "real_only_missing_zero":
        return np.where(observed, values, 0.0)
    if condition == "imputed_only_real_zero":
        return np.where(observed, 0.0, medians)
    raise ValueError("Strict overlap uses its verified fold-specific model-fitting pathway.")


In [ ]:
aggregate_path = REPO_ROOT / "results" / "aggregate" / "radiomics_imputation_controls.csv"
if not aggregate_path.is_file():
    raise FileNotFoundError("Released aggregate radiomics control table is unavailable.")
controls = pd.read_csv(aggregate_path)
controls


## Verified aggregate findings

HCC sensitivity was 0.961 for real-plus-imputed, 1.000 for median-only, 0.243 for strict overlap,
0.010 for real-only with unavailable cells zeroed, and 0.971 for imputed-only with observed cells
zeroed. The contrast indicates that apparent external W5 sensitivity depended strongly on the
imputation pathway. Condition C remains an internal diagnostic refit and cannot be interpreted as
frozen-model external performance.
